Map Reduce Style Implementation using Github Repo

In [1]:
!pip install -q pyspark

In [2]:
!git clone https://github.com/unaizahhhhh/MIT805-Group-4-Assignment.git

Cloning into 'MIT805-Group-4-Assignment'...
remote: Enumerating objects: 165, done.
remote: Counting objects: 100% (4/4), done.
remote: Compressing objects: 100% (4/4), done.
remote: Total 165 (delta 0), reused 3 (delta 0), pack-reused 161 (from 2)
Receiving objects: 100% (165/165), 1.55 GiB | 26.90 MiB/s, done.
Resolving deltas: 100% (7/7), done.
Updating files: 100% (136/136), done.


In [48]:
!cd MIT805-Group-4-Assignment && git pull

Already up to date.


In [49]:
!ls MIT805-Group-4-Assignment/data/

flight_with_weather_2016  flight_with_weather_2020  flight_with_weather_2024
flight_with_weather_2017  flight_with_weather_2021  README.md
flight_with_weather_2018  flight_with_weather_2022
flight_with_weather_2019  flight_with_weather_2023


In [50]:
!ls MIT805-Group-4-Assignment/
!ls MIT805-Group-4-Assignment/src/

data  figures  notebooks  README.md  report  requirements.txt  src
aggregate.py  __init__.py  __pycache__	README.md  transform.py


In [6]:
!du -sh MIT805-Group-4-Assignment/data/

2.4G	MIT805-Group-4-Assignment/data/


In [7]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("FlightDelayMapReduce") \
    .getOrCreate()

spark

In [8]:
YEARS = range(2016, 2025)
paths = [f"MIT805-Group-4-Assignment/data/flight_with_weather_{y}/" for y in YEARS]

df_all = spark.read.parquet(*paths)

df_all.printSchema()
print(f"Partitions: {df_all.rdd.getNumPartitions()}")

root
 |-- FL_DATE: timestamp (nullable = true)
 |-- OP_CARRIER: string (nullable = true)
 |-- OP_CARRIER_FL_NUM: double (nullable = true)
 |-- ORIGIN: string (nullable = true)
 |-- DEST: string (nullable = true)
 |-- CRS_DEP_TIME: timestamp (nullable = true)
 |-- DEP_TIME: timestamp (nullable = true)
 |-- DEP_DELAY: double (nullable = true)
 |-- TAXI_OUT: double (nullable = true)
 |-- WHEELS_OFF: timestamp (nullable = true)
 |-- WHEELS_ON: timestamp (nullable = true)
 |-- TAXI_IN: double (nullable = true)
 |-- CRS_ARR_TIME: timestamp (nullable = true)
 |-- ARR_TIME: timestamp (nullable = true)
 |-- ARR_DELAY: double (nullable = true)
 |-- CRS_ELAPSED_TIME: double (nullable = true)
 |-- ACTUAL_ELAPSED_TIME: double (nullable = true)
 |-- AIR_TIME: double (nullable = true)
 |-- FLIGHTS: double (nullable = true)
 |-- MONTH: integer (nullable = true)
 |-- DAY_OF_MONTH: integer (nullable = true)
 |-- DAY_OF_WEEK: integer (nullable = true)
 |-- ORIGIN_INDEX: integer (nullable = true)
 |-- DES

In [9]:
df_all.printSchema()
print(f"Rows: {df_all.count():,}")
print(f"Columns: {len(df_all.columns)}")
print(df_all.columns)

root
 |-- FL_DATE: timestamp (nullable = true)
 |-- OP_CARRIER: string (nullable = true)
 |-- OP_CARRIER_FL_NUM: double (nullable = true)
 |-- ORIGIN: string (nullable = true)
 |-- DEST: string (nullable = true)
 |-- CRS_DEP_TIME: timestamp (nullable = true)
 |-- DEP_TIME: timestamp (nullable = true)
 |-- DEP_DELAY: double (nullable = true)
 |-- TAXI_OUT: double (nullable = true)
 |-- WHEELS_OFF: timestamp (nullable = true)
 |-- WHEELS_ON: timestamp (nullable = true)
 |-- TAXI_IN: double (nullable = true)
 |-- CRS_ARR_TIME: timestamp (nullable = true)
 |-- ARR_TIME: timestamp (nullable = true)
 |-- ARR_DELAY: double (nullable = true)
 |-- CRS_ELAPSED_TIME: double (nullable = true)
 |-- ACTUAL_ELAPSED_TIME: double (nullable = true)
 |-- AIR_TIME: double (nullable = true)
 |-- FLIGHTS: double (nullable = true)
 |-- MONTH: integer (nullable = true)
 |-- DAY_OF_MONTH: integer (nullable = true)
 |-- DAY_OF_WEEK: integer (nullable = true)
 |-- ORIGIN_INDEX: integer (nullable = true)
 |-- DES

In [10]:
df_all.show(5, truncate=False)

+-------------------+----------+-----------------+------+----+-------------------+-------------------+---------+--------+-------------------+-------------------+-------+-------------------+-------------------+---------+----------------+-------------------+--------+-------+-----+------------+-----------+------------+----------+------+------+------+------+------+------+----------+-----------+----------+-----------+
|FL_DATE            |OP_CARRIER|OP_CARRIER_FL_NUM|ORIGIN|DEST|CRS_DEP_TIME       |DEP_TIME           |DEP_DELAY|TAXI_OUT|WHEELS_OFF         |WHEELS_ON          |TAXI_IN|CRS_ARR_TIME       |ARR_TIME           |ARR_DELAY|CRS_ELAPSED_TIME|ACTUAL_ELAPSED_TIME|AIR_TIME|FLIGHTS|MONTH|DAY_OF_MONTH|DAY_OF_WEEK|ORIGIN_INDEX|DEST_INDEX|O_TEMP|O_PRCP|O_WSPD|D_TEMP|D_PRCP|D_WSPD|O_LATITUDE|O_LONGITUDE|D_LATITUDE|D_LONGITUDE|
+-------------------+----------+-----------------+------+----+-------------------+-------------------+---------+--------+-------------------+-------------------+-----

In [11]:
df_all.describe().show()

+-------+----------+------------------+--------+--------+------------------+-----------------+-----------------+-----------------+------------------+-------------------+------------------+--------+-----------------+------------------+------------------+------------------+-----------------+------------------+-------------------+------------------+-----------------+-------------------+------------------+----------------+------------------+-----------------+------------------+
|summary|OP_CARRIER| OP_CARRIER_FL_NUM|  ORIGIN|    DEST|         DEP_DELAY|         TAXI_OUT|          TAXI_IN|        ARR_DELAY|  CRS_ELAPSED_TIME|ACTUAL_ELAPSED_TIME|          AIR_TIME| FLIGHTS|            MONTH|      DAY_OF_MONTH|       DAY_OF_WEEK|      ORIGIN_INDEX|       DEST_INDEX|            O_TEMP|             O_PRCP|            O_WSPD|           D_TEMP|             D_PRCP|            D_WSPD|      O_LATITUDE|       O_LONGITUDE|       D_LATITUDE|       D_LONGITUDE|
+-------+----------+------------------+---

In [12]:
from pyspark.sql import functions as F

df_all.select([
    F.count(F.when(F.col(c).isNull(), c)).alias(c) for c in df_all.columns
]).show()

+-------+----------+-----------------+------+----+------------+--------+---------+--------+----------+---------+-------+------------+--------+---------+----------------+-------------------+--------+-------+-----+------------+-----------+------------+----------+------+------+------+------+------+------+----------+-----------+----------+-----------+
|FL_DATE|OP_CARRIER|OP_CARRIER_FL_NUM|ORIGIN|DEST|CRS_DEP_TIME|DEP_TIME|DEP_DELAY|TAXI_OUT|WHEELS_OFF|WHEELS_ON|TAXI_IN|CRS_ARR_TIME|ARR_TIME|ARR_DELAY|CRS_ELAPSED_TIME|ACTUAL_ELAPSED_TIME|AIR_TIME|FLIGHTS|MONTH|DAY_OF_MONTH|DAY_OF_WEEK|ORIGIN_INDEX|DEST_INDEX|O_TEMP|O_PRCP|O_WSPD|D_TEMP|D_PRCP|D_WSPD|O_LATITUDE|O_LONGITUDE|D_LATITUDE|D_LONGITUDE|
+-------+----------+-----------------+------+----+------------+--------+---------+--------+----------+---------+-------+------------+--------+---------+----------------+-------------------+--------+-------+-----+------------+-----------+------------+----------+------+------+------+------+------+----

Running the Map stage

In [13]:
import sys
sys.path.append("MIT805-Group-4-Assignment/src")

from transform import clean_and_derive

df_clean = clean_and_derive(df_all)
df_clean.cache()

df_clean.printSchema()
print(f"Total rows: {df_clean.count():,}")
df_clean.show(5, truncate=False)

root
 |-- FL_DATE: date (nullable = true)
 |-- OP_CARRIER: string (nullable = true)
 |-- ORIGIN: string (nullable = true)
 |-- DEST: string (nullable = true)
 |-- DEP_DELAY: double (nullable = true)
 |-- ARR_DELAY: double (nullable = true)
 |-- O_PRCP: double (nullable = true)
 |-- D_PRCP: double (nullable = true)
 |-- O_TEMP: double (nullable = true)
 |-- D_TEMP: double (nullable = true)
 |-- O_WSPD: double (nullable = true)
 |-- D_WSPD: double (nullable = true)
 |-- YEAR: integer (nullable = true)
 |-- MONTH: integer (nullable = true)
 |-- DAY_OF_WEEK: string (nullable = true)
 |-- ROUTE: string (nullable = false)
 |-- DELAYED_15: integer (nullable = false)
 |-- SEVERE_DELAY_60: integer (nullable = false)
 |-- PRECIPITATION_CATEGORY: string (nullable = false)

Total rows: 54,674,003
+----------+----------+------+----+---------+---------+------+------+------+------+------+------+----+-----+-----------+-------+----------+---------------+----------------------+
|FL_DATE   |OP_CARRIER|OR

In [47]:
from aggregate import overall_summary, carrier_delay_summary, weather_delay_comparison, yearly_trend, weather_correlation, day_of_week_summary, origin_airport_summary, carrier_delay_percentiles, delay_by_carrier_rdd

ImportError: cannot import name 'delay_by_carrier_rdd' from 'aggregate' (/content/MIT805-Group-4-Assignment/src/aggregate.py)

In [16]:
overall_summary(df_clean).show(truncate=False)

+-------------+---------------+-------------+-------------+----------------+-----------------------+
|total_flights|delayed_flights|severe_delays|average_delay|delay_percentage|severe_delay_percentage|
+-------------+---------------+-------------+-------------+----------------+-----------------------+
|54674003     |10116916       |3360584      |4.48         |18.5            |6.15                   |
+-------------+---------------+-------------+-------------+----------------+-----------------------+



In [17]:
carrier_delay_summary(df_clean).show(20, truncate=False)

+----------+-------------+---------------+-------------+----------------+
|OP_CARRIER|total_flights|delayed_flights|average_delay|delay_percentage|
+----------+-------------+---------------+-------------+----------------+
|B6        |2236608      |590147         |11.34        |26.39           |
|F9        |1176549      |305928         |11.03        |26.0            |
|VX        |155092       |39008          |8.44         |25.15           |
|G4        |303824       |76198          |12.31        |25.08           |
|NK        |1696082      |371387         |7.54         |21.9            |
|AA        |7518473      |1532817        |7.0          |20.39           |
|EV        |1178841      |231615         |7.4          |19.65           |
|WN        |11133643     |2151883        |3.85         |19.33           |
|YV        |818626       |158168         |7.97         |19.32           |
|UA        |5083571      |948076         |4.35         |18.65           |
|OH        |1538083      |284175      

In [18]:
route_weather = weather_delay_comparison(df_clean, "ROUTE").filter(
    (F.col("dry_flights") >= 3000) & (F.col("rain_flights") >= 500)
)
route_weather.show(20, truncate=False)

+-------+-----------+--------------------+-----------------+------------+---------------------+------------------+-------------------------+----------------------+
|ROUTE  |dry_flights|dry_delay_percentage|dry_average_delay|rain_flights|rain_delay_percentage|rain_average_delay|delay_percentage_increase|average_delay_increase|
+-------+-----------+--------------------+-----------------+------------+---------------------+------------------+-------------------------+----------------------+
|PSP-SFO|20181      |21.14               |7.34             |1471        |49.42                |40.29             |28.28                    |32.95                 |
|HDN-DEN|6843       |19.16               |1.94             |877         |47.43                |31.8              |28.27                    |29.86                 |
|DEN-COS|24968      |21.88               |9.25             |1630        |48.53                |36.58             |26.65                    |27.33                 |
|LAS-SFO|53515  

In [19]:
carrier_weather = weather_delay_comparison(df_clean, "OP_CARRIER")
carrier_weather.show(20, truncate=False)

+----------+-----------+--------------------+-----------------+------------+---------------------+------------------+-------------------------+----------------------+
|OP_CARRIER|dry_flights|dry_delay_percentage|dry_average_delay|rain_flights|rain_delay_percentage|rain_average_delay|delay_percentage_increase|average_delay_increase|
+----------+-----------+--------------------+-----------------+------------+---------------------+------------------+-------------------------+----------------------+
|EV        |1015918    |17.55               |4.98             |156781      |33.33                |23.04             |15.78                    |18.06                 |
|MQ        |1518919    |16.31               |2.08             |228173      |32.0                 |17.1              |15.69                    |15.02                 |
|VX        |137370     |23.37               |6.51             |17717       |39.0                 |23.4              |15.63                    |16.89                 

In [20]:
yearly_trend(df_clean).show(9)

+----+-------------+-------------+----------------+
|YEAR|total_flights|average_delay|delay_percentage|
+----+-------------+-------------+----------------+
|2016|      5537987|         3.52|           17.41|
|2017|      5575872|         4.32|           18.45|
|2018|      6986842|          5.0|           19.09|
|2019|      7161827|         5.38|           19.11|
|2020|      4312091|        -5.11|            9.73|
|2021|      5755666|         2.95|           17.08|
|2022|      6413416|         6.93|           20.99|
|2023|      6645461|         6.57|           20.54|
|2024|      6284841|         7.11|           20.81|
+----+-------------+-------------+----------------+



In [21]:
weather_correlation(df_clean).show(truncate=False)


+------------------------+----------------------+----------------------+----------------------+--------------------+
|corr_origin_precip_delay|corr_origin_wind_delay|corr_origin_temp_delay|corr_dest_precip_delay|corr_dest_wind_delay|
+------------------------+----------------------+----------------------+----------------------+--------------------+
|0.010213709329053224    |0.04718847801392416   |0.04941348035155309   |0.01219679217315569   |0.05133903270781939 |
+------------------------+----------------------+----------------------+----------------------+--------------------+



In [22]:
day_of_week_summary(df_clean).show()


+-----------+-------------+-------------+----------------+
|DAY_OF_WEEK|total_flights|average_delay|delay_percentage|
+-----------+-------------+-------------+----------------+
|     Friday|      8145574|         6.43|           20.45|
|   Thursday|      8093665|         5.77|           19.81|
|     Sunday|      7867557|         5.11|           19.09|
|     Monday|      8142491|         5.13|           18.92|
|  Wednesday|      7793816|         2.95|           17.13|
|   Saturday|      6906200|         3.08|           17.11|
|    Tuesday|      7724700|         2.52|           16.68|
+-----------+-------------+-------------+----------------+



In [23]:
origin_airport_summary(df_clean).show(20)


+------+-------------+-----------------+----------------+
|ORIGIN|total_flights|average_dep_delay|delay_percentage|
+------+-------------+-----------------+----------------+
|   SWF|         6317|            19.89|           27.96|
|   BQN|        16975|            18.84|           27.76|
|   ASE|        49715|            20.62|           26.99|
|   MVY|         5870|            20.21|           26.59|
|   ORH|         8566|            16.55|           24.98|
|   TTN|        18275|            15.12|           24.49|
|   ACK|         9923|            18.44|           24.34|
|   PSE|         6228|            11.17|           24.02|
|   EWR|      1020150|            14.86|            23.6|
|   FLL|       756499|             14.1|           23.02|
|   MIA|       755575|            13.61|           22.85|
|   HDN|        14363|            12.79|           22.63|
|   EGE|        20808|            18.05|           22.31|
|   DFW|      2245176|            12.95|           22.26|
|   MCO|      

In [40]:
carrier_delay_percentiles(df_clean).show(20)

+----------+------------+---------+----------+
|OP_CARRIER|median_delay|p90_delay|mean_delay|
+----------+------------+---------+----------+
|        HA|        -2.0|     20.0|      2.82|
|        VX|        -2.0|     50.0|      8.44|
|        G4|        -3.0|     53.0|     12.31|
|        QX|        -4.0|     23.0|      2.33|
|        WN|        -5.0|     33.0|      3.85|
|        B6|        -5.0|     64.0|     11.34|
|        F9|        -5.0|     59.0|     11.03|
|        NK|        -6.0|     47.0|      7.54|
|        YV|        -6.0|     44.0|      7.97|
|        AA|        -6.0|     41.0|       7.0|
|        MQ|        -6.0|     35.0|      4.04|
|        OH|        -6.0|     38.0|      5.48|
|        AS|        -6.0|     28.0|      0.87|
|        EV|        -7.0|     46.0|       7.4|
|        OO|        -7.0|     35.0|      5.11|
|        UA|        -8.0|     39.0|      4.35|
|        DL|        -8.0|     25.0|      0.66|
|        YX|       -10.0|     31.0|      0.26|
|        9E| 

In [52]:
import importlib
import aggregate
importlib.reload(aggregate)

from aggregate import delay_by_carrier_rdd

In [53]:
from aggregate import delay_by_carrier_rdd

rdd_result = delay_by_carrier_rdd(df_clean)
rdd_sorted = sorted(rdd_result.collect(), key=lambda x: -x[1])

for carrier, avg_delay in rdd_sorted[:10]:
    print(f"{carrier}: {avg_delay} min avg delay")

G4: 12.31 min avg delay
B6: 11.34 min avg delay
F9: 11.03 min avg delay
VX: 8.44 min avg delay
YV: 7.97 min avg delay
NK: 7.54 min avg delay
EV: 7.4 min avg delay
AA: 7.0 min avg delay
OH: 5.48 min avg delay
OO: 5.11 min avg delay
